# 04 — The learned arm: a siamese CNN on before/after pixel chips

**Question this notebook answers:** does looking at the *pixels* — spatial
texture and pattern in before/after chips — beat the plot-mean time series,
and what actually moves the needle: the architecture, or the training data?

**Task formulation, defended.** Per plot: a 6-band chip at the EUDR cutoff
epoch (T1, dry season 2020) and one recent (T2, dry season 2024) → one
probability that clearing occurred between them.
- *Why chip classification, not segmentation:* the compliance decision unit
  is the plot ("did clearing occur on it"), the labels are 30 m weak labels
  whose edges are exactly what a 10 m segmentation loss would fixate on, and
  a classification model trains in minutes on a free T4. Segmentation is the
  natural extension, not the right first system.
- *Why bi-temporal input, not just the recent image:* a single-date model
  learns "what does cleared land look like" and flags every plot that was
  *already* pasture in 2020 — precisely the plots EUDR does not care about.
  The pair makes "was forest, became not-forest" learnable.
- *Why a siamese encoder:* both dates are the same sensor over the same land,
  so they should be described by the same features — sharing weights enforces
  that and halves the parameters. The head sees `[f2−f1, f1, f2]`: the
  difference carries *what changed*; the absolutes let it condition on what
  it changed *from*.
- *Why from scratch:* 6-band multispectral input has no honest ImageNet
  initialisation, and thousands of chips are enough for a ~200k-parameter
  model. Geospatial foundation models are the obvious next arm — discussed
  at the end, not silently skipped.

**The experiment that matters most here** is not the architecture at all:
it is the **hard-negative ablation**. Negatives sampled only from our plot
portfolio under-represent the confusable cases (textured stable forest).
We mine additional stable-forest negatives from the JRC Tropical Moist
Forest product and train with and without them. If precision moves more
than any architecture knob would — that is the finding.

**Produces**
- `outputs/chips.npz` (cached), `outputs/cnn_seed*.pt`, `outputs/cnn_predictions.csv`
- `figures/g04_*.png`

**Expected runtime:** first run ~45 min (chip downloads dominate; cached
afterwards) + ~10 min training on a T4. **Runtime → Change runtime type →
T4 GPU** before starting.

### Environment setup and persistence

On Colab this clones the repository, installs dependencies, and mounts Google
Drive so that **outputs survive the session**. Locally it is a no-op beyond
putting `src/` on the path.

**Why Drive.** A Colab VM is deleted when the session ends, and the notebooks
depend on each other's artefacts: 01 writes the validated plot portfolio that
every later chapter loads; 02 writes the forest baselines; 03 the time series;
04 the model predictions. `outputs/`, `figures/` and `evidence/` are therefore
redirected to Drive via environment variables that `geoverdict.config` reads
at import time — which is why they must be set *before* the import.

**Re-running this cell picks up code changes**: it hard-resets the clone to
`origin/main` and purges `geoverdict` from `sys.modules` (Python caches
imports; a `git pull` alone leaves the kernel running the old code). Treat the
clone as read-only — edit code locally and push, not inside `/content`.

In [ ]:
# --- edit these if you are running your own fork ---------------------------
GITHUB_USER = "SaadH-077"
USE_DRIVE = True          # False -> everything stays in the ephemeral session
# ---------------------------------------------------------------------------

import os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = "geoverdict"

if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        PERSIST = Path("/content/drive/MyDrive/geoverdict")
        for sub in ("outputs", "figures", "outputs/evidence"):
            (PERSIST / sub).mkdir(parents=True, exist_ok=True)
        os.environ["GEOVERDICT_OUTPUT_DIR"] = str(PERSIST / "outputs")
        os.environ["GEOVERDICT_FIGURE_DIR"] = str(PERSIST / "figures")
        os.environ["GEOVERDICT_EVIDENCE_DIR"] = str(PERSIST / "outputs" / "evidence")
        print("persisting outputs and figures to", PERSIST)

    if not Path(REPO).exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        f"https://github.com/{GITHUB_USER}/{REPO}.git"], check=False)
    if Path(REPO).exists():
        os.chdir(REPO)
        subprocess.run(["git", "fetch", "--quiet", "--depth", "50", "origin", "main"], check=False)
        before = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
        subprocess.run(["git", "reset", "--hard", "--quiet", "origin/main"], check=False)
        after = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
        if before != after:
            print(f"repo updated {before[:7]} -> {after[:7]}")
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    except subprocess.CalledProcessError as exc:
        print("!! dependency install failed:", exc)
        print("!! continuing anyway - the cells below will report what is missing")

ROOT = Path.cwd() if Path.cwd().name == REPO or (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

# Purge cached geoverdict modules so a repo update takes effect in THIS kernel.
for m in [m for m in list(sys.modules) if m == "geoverdict" or m.startswith("geoverdict.")]:
    del sys.modules[m]

from geoverdict import config as cfg
from geoverdict import viz

cfg.ensure_dirs()
viz.set_style()
print(f"outputs -> {cfg.OUTPUT_DIR}")
print(f"figures -> {cfg.FIGURE_DIR}")
print(f"seed = {cfg.SEED}, AOI = {cfg.AOI_NAME} {cfg.AOI_BBOX}")

In [ ]:
EE_PROJECT = ""   # <- same Earth Engine project id as before (for hard negatives)

import geopandas as gpd
import numpy as np
import pandas as pd

plots = gpd.read_file(cfg.OUTPUT_DIR / "plots_analysis.geojson")
baseline = pd.read_csv(cfg.OUTPUT_DIR / "baseline.csv", dtype={"plot_id": str})
plots = plots.merge(baseline, on="plot_id", suffixes=("", "_b"))

loss = plots.hansen_loss_post_frac.fillna(0)
plots["label"] = np.where(loss > cfg.POS_LOSS_FRAC, 1,
                  np.where(loss < cfg.NEG_LOSS_FRAC, 0, -1))
print(plots.label.value_counts().rename({1: "positive (cleared)", 0: "negative (stable)",
                                         -1: "ambiguous (excluded from training)"}))

### Labels, and the ambiguous band we refuse to train on

Weak labels from Hansen post-2020 loss: **> 20% of the plot lost → positive**,
**< 2% → negative**, and the 2–20% band is *excluded from training* — those
labels are guesses, and training on labels you do not trust in the ambiguous
band injects exactly the noise you cannot diagnose later. Excluded plots are
still screened at inference; exclusion is a training decision, not a coverage
hole.

### Hard negatives, mined from TMF

The portfolio's own negatives are a biased sample of "no change": many were
never forest at all (easy — bright, smooth pasture). The negatives that teach
the actual decision boundary are **undisturbed tropical moist forest**: dark,
textured, high-NDVI chips where nothing happened — spectrally adjacent to a
pre-clearing T1. We sample those locations from the JRC TMF product via Earth
Engine and add them as extra negative chips.

In [ ]:
from geoverdict import gee
gee.init(project=EE_PROJECT or None)

N_HARD = 150
hard_pts = gee.stable_forest_mask_points(cfg.AOI_BBOX, N_HARD, seed=cfg.SEED)
print(f"mined {len(hard_pts)} stable-forest hard-negative locations "
      f"(TMF asset: {hard_pts.attrs.get('assets')})")

### Chips via STAC windowed reads (cached to Drive)

For chips we switch to the STAC/COG path (`geoverdict.s2`): we need the
actual pixels and full control over masking, and it is the access path a
production system owns end-to-end. Per sample and per epoch (T1/T2) we try
the three least-cloudy dry-season scenes in order and keep the first chip
that is ≥ 70% valid — per-plot cloud luck differs even within one scene, so
scene-level selection alone is not enough.

This is the slow cell (~10⁴ windowed HTTP reads). It writes `chips.npz` to
Drive; every later run loads the cache in seconds.

In [ ]:
from geoverdict import s2

chip_cache = cfg.OUTPUT_DIR / "chips.npz"
if chip_cache.exists():
    z = np.load(chip_cache, allow_pickle=True)
    X1, X2 = z["x1"], z["x2"]
    meta = pd.DataFrame({k: z[k] for k in ("sample_id", "kind", "label")})
    t1_ids, t2_ids = z["t1_items"].tolist(), z["t2_items"].tolist()
    print(f"loaded chip cache: {X1.shape}")
else:
    items = s2.search_items()
    print(f"STAC: {len(items)} L2A items over the AOI 2019-2025")

    def dry_candidates(window, k=3):
        import pandas as pd
        s, e = pd.Timestamp(window[0]), pd.Timestamp(window[1])
        c = [it for it in items if s <= pd.Timestamp(str(it.datetime)).tz_localize(None) <= e]
        return sorted(c, key=lambda it: it.properties.get("eo:cloud_cover", 100))[:k]

    cand_t1, cand_t2 = dry_candidates(cfg.T1_WINDOW), dry_candidates(cfg.T2_WINDOW)
    print("T1 candidates:", [(it.id, round(it.properties.get('eo:cloud_cover', -1), 1)) for it in cand_t1])
    print("T2 candidates:", [(it.id, round(it.properties.get('eo:cloud_cover', -1), 1)) for it in cand_t2])

    samples = [{"sample_id": r.plot_id, "kind": "plot",
                "lon": r.geometry.centroid.x, "lat": r.geometry.centroid.y,
                "label": int(r.label)} for r in plots.itertuples()]
    samples += [{"sample_id": f"hn{i}", "kind": "hard_negative",
                 "lon": p.lon, "lat": p.lat, "label": 0}
                for i, p in enumerate(hard_pts.itertuples())]

    def best_chip(cands, lon, lat):
        for it in cands:
            obs = s2.observe_plot(it, lon, lat, keep_chip=True)
            if obs is not None and obs.valid_frac >= 0.70:
                return obs
        return None

    rows, x1s, x2s = [], [], []
    from tqdm.auto import tqdm
    for smp in tqdm(samples):
        o1 = best_chip(cand_t1, smp["lon"], smp["lat"])
        o2 = best_chip(cand_t2, smp["lon"], smp["lat"])
        if o1 is None or o2 is None:
            continue
        rows.append({**smp, "t1_item": o1.item_id, "t2_item": o2.item_id})
        x1s.append(o1.chip); x2s.append(o2.chip)

    X1, X2 = np.stack(x1s), np.stack(x2s)
    meta = pd.DataFrame(rows)
    t1_ids = sorted(meta.t1_item.unique().tolist())
    t2_ids = sorted(meta.t2_item.unique().tolist())
    np.savez_compressed(chip_cache, x1=X1, x2=X2,
                        sample_id=meta.sample_id.to_numpy(), kind=meta.kind.to_numpy(),
                        label=meta.label.to_numpy(),
                        t1_items=np.array(t1_ids), t2_items=np.array(t2_ids))
    print(f"chips: {X1.shape}, coverage {len(meta)}/{len(samples)} samples "
          f"({1 - len(meta)/len(samples):.0%} lost to cloud, reported not hidden)")

In [ ]:
import matplotlib.pyplot as plt

# a look at the data the model will see: cleared vs stable pairs
lab = meta.label.to_numpy()
show = np.concatenate([np.where(lab == 1)[0][:4], np.where(lab == 0)[0][:4]])
fig, axes = plt.subplots(4, 4, figsize=(11, 11))
for k, i in enumerate(show[:8]):
    r, c = divmod(k, 2)
    viz.before_after_panel((axes[r, c*2], axes[r, c*2+1]), X1[i], X2[i],
                           f"{meta.sample_id.iloc[i]} T1", f"T2 (label={lab[i]})")
fig.suptitle("What clearing looks like from 786 km: T1 (2020) vs T2 (2024)", fontweight="bold")
fig.tight_layout()
viz.save(fig, "g04_chip_examples")
plt.show()

### Spatially-blocked splits, then training (3 seeds)

Longitude thirds again: **train | val | test = west | middle | east**, hard
negatives assigned by the same rule. Normalisation statistics come from the
training block only (statistics computed on data you evaluate on is quiet
leakage). Class imbalance is handled in the loss (`pos_weight = n_neg/n_pos`)
rather than by oversampling — with weak labels, duplicating positives also
duplicates their label errors.

Three seeds, mean ± std reported: a single-seed number on a few hundred test
chips is an anecdote with a decimal point.

In [ ]:
import torch
from geoverdict import models as M
from geoverdict import metrics as MT

train_mask_all = np.zeros(len(meta), dtype=bool)
lon_all = np.array([plots.set_index("plot_id").geometry.centroid.x.get(s, np.nan)
                    for s in meta.sample_id])
# hard negatives carry their own lon
hn = meta.kind.to_numpy() == "hard_negative"
lon_all[hn] = hard_pts.lon.to_numpy()[
    [int(s[2:]) for s in meta.sample_id[hn]]]
q1, q2 = np.nanquantile(lon_all, [1/3, 2/3])
split = np.where(lon_all <= q1, "train", np.where(lon_all <= q2, "val", "test"))

usable = lab >= 0
idx = {name: np.where(usable & (split == name))[0] for name in ("train", "val", "test")}
print({k: (len(v), int(lab[v].sum())) for k, v in idx.items()},
      "(n, positives) per block")

stats = {"mean": X1[idx["train"]].mean(axis=(0, 2, 3)),
         "std": X1[idx["train"]].std(axis=(0, 2, 3)) + 1e-6}

def make_loader(ids, augment, batch=64, seed=0):
    ds = M.ChipPairDataset(X1[ids], X2[ids], lab[ids].astype(float),
                           stats=stats, augment=augment, seed=seed)
    return torch.utils.data.DataLoader(ds, batch_size=batch, shuffle=augment)

pos_weight = float((lab[idx["train"]] == 0).sum() / max((lab[idx["train"]] == 1).sum(), 1))
print(f"pos_weight = {pos_weight:.1f}")

histories, test_logits = [], []
for seed in cfg.SEEDS:
    cfg.set_seed(seed)
    model = M.SiameseChangeNet()
    hist = M.fit(model, make_loader(idx["train"], True, seed=seed),
                 make_loader(idx["val"], False),
                 epochs=40, pos_weight=pos_weight)
    histories.append(hist)
    _, zl, _ = M.evaluate(model, make_loader(idx["test"], False))
    test_logits.append(zl)
    torch.save(model.state_dict(), cfg.OUTPUT_DIR / f"cnn_seed{seed}.pt")
    print(f"seed {seed}: best val PR-AUC {hist.best_val_pr_auc:.3f} @ epoch {hist.best_epoch}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for h, seed in zip(histories, cfg.SEEDS):
    axes[0].plot(h.train_loss, lw=1, label=f"train s{seed}")
    axes[0].plot(h.val_loss, lw=1, ls="--", label=f"val s{seed}")
    axes[1].plot(h.val_pr_auc, lw=1.4, label=f"seed {seed}")
axes[0].set_title("loss"); axes[0].set_xlabel("epoch"); axes[0].legend(fontsize=7)
axes[1].set_title("validation PR-AUC (early-stop criterion)")
axes[1].set_xlabel("epoch"); axes[1].legend(fontsize=8)
viz.save(fig, "g04_training")
plt.show()

### The three-arm comparison, on identical test plots

One PR plot, three decision rules on the same information budget:
the transparent detector (chapter 03), the random forest on temporal
features, and the CNN on pixels. Read it as a question about *marginal
value*: what does each increment of model complexity buy, in precision at
the recall a compliance policy actually demands?

In [ ]:
from sklearn.metrics import precision_recall_curve

y_test = lab[idx["test"]].astype(int)
cnn_probs = 1 / (1 + np.exp(-np.mean(test_logits, axis=0)))

# align chapter-03 arms onto the same test plots (plots only, no hard negatives)
det = pd.read_csv(cfg.OUTPUT_DIR / "ts_detections.csv", dtype={"plot_id": str}).set_index("plot_id")
test_sids = meta.sample_id.to_numpy()[idx["test"]]
is_plot = np.array([s in det.index for s in test_sids])
ts_scores = np.array([det.score.get(s, 0.0) for s in test_sids])

curves = {}
for name, scores, mask in (("statistics arm (ch.03)", ts_scores, is_plot),
                           ("siamese CNN (pixels)", cnn_probs, np.ones_like(is_plot, bool))):
    yt, sc = y_test[mask], scores[mask]
    p, r, _ = precision_recall_curve(yt, sc)
    curves[name] = (p, r, MT.pr_auc(yt, sc))

fig, ax = plt.subplots(figsize=(6.5, 5))
viz.plot_pr_curves(ax, curves)
ax.set_title("Same plots, increasing model complexity")
viz.save(fig, "g04_pr_comparison")
plt.show()

screen = pd.DataFrame(MT.screening_table(y_test, cnn_probs))
print(screen.round(3).to_string(index=False))
cfg.append_result({"notebook": "04", "name": "cnn_eval",
                   "pr_auc_mean": float(np.mean([MT.pr_auc(y_test, 1/(1+np.exp(-z)))
                                                 for z in test_logits])),
                   "pr_auc_std": float(np.std([MT.pr_auc(y_test, 1/(1+np.exp(-z)))
                                               for z in test_logits])),
                   "flags_per_1000_at_r90": float(screen.loc[screen.recall_target == 0.90,
                                                             "flags_per_1000"].iloc[0]),
                   "n_test": int(len(y_test)), "seeds": list(cfg.SEEDS)})

### The headline experiment: what do hard negatives buy?

Retrain (one seed — the deltas here dwarf seed noise, and the multi-seed
main result is already banked) with the TMF stable-forest negatives
**removed** from training. Test set unchanged, including its hard negatives
— deployment does not remove the confusable forest from the world.

The hypothesis, stated before the result: without hard negatives the model
meets textured dark forest at test time having rarely seen it labelled
"no change", and **precision** is what should collapse — false alarms on
stable forest — while recall barely moves.

In [ ]:
no_hn = idx["train"][meta.kind.to_numpy()[idx["train"]] != "hard_negative"]
cfg.set_seed(cfg.SEEDS[0])
model_nohn = M.SiameseChangeNet()
_ = M.fit(model_nohn, make_loader(no_hn, True), make_loader(idx["val"], False),
          epochs=40, pos_weight=float((lab[no_hn] == 0).sum() / max((lab[no_hn] == 1).sum(), 1)))
_, z_nohn, _ = M.evaluate(model_nohn, make_loader(idx["test"], False))
p_nohn = 1 / (1 + np.exp(-z_nohn))

rows = []
for name, sc in (("with hard negatives", cnn_probs), ("without", p_nohn)):
    thr = MT.threshold_at_recall(y_test, sc, 0.90)
    m = MT.prf(y_test, sc >= thr)
    rows.append({"training": name, "precision@r90": m["precision"],
                 "recall": m["recall"], "pr_auc": MT.pr_auc(y_test, sc)})
abl = pd.DataFrame(rows)
print(abl.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(abl.training, abl["precision@r90"],
       color=[viz.PALETTE["forest"], viz.PALETTE["clearing"]])
for x, v in enumerate(abl["precision@r90"]):
    ax.text(x, v, f"{v:.2f}", ha="center", va="bottom", fontweight="bold")
ax.set_ylabel("precision at 90% recall")
ax.set_title("The data, not the architecture:\nwhat stable-forest hard negatives buy")
viz.save(fig, "g04_hard_negative_ablation")
plt.show()
cfg.append_result({"notebook": "04", "name": "hard_negative_ablation",
                   "precision_at_r90_with": float(abl["precision@r90"][0]),
                   "precision_at_r90_without": float(abl["precision@r90"][1])})

### Calibration — because chapter 05 consumes these probabilities

The verdict layer treats p ≥ 0.7 as corroborating evidence. That sentence is
only meaningful if 0.7 *means* 70%. We fit one temperature on the validation
block (fitting the calibrator on the set you then report is the same crime
as testing on train) and show reliability before/after on test. The
calibrated probabilities are what get saved for chapter 05.

In [ ]:
val_logits = []
for seed in cfg.SEEDS:
    model = M.SiameseChangeNet()
    model.load_state_dict(torch.load(cfg.OUTPUT_DIR / f"cnn_seed{seed}.pt",
                                     map_location="cpu"))
    _, zv, _ = M.evaluate(model, make_loader(idx["val"], False))
    val_logits.append(zv)
zv_mean = np.mean(val_logits, axis=0)
T = MT.fit_temperature(zv_mean, lab[idx["val"]].astype(float))
print(f"fitted temperature T = {T:.2f}  (T > 1 means the raw model was overconfident)")

z_test = np.mean(test_logits, axis=0)
rep_raw = MT.expected_calibration_error(1/(1+np.exp(-z_test)), y_test)
rep_cal = MT.expected_calibration_error(1/(1+np.exp(-z_test/T)), y_test)

fig, ax = plt.subplots(figsize=(5.5, 5))
viz.plot_reliability(ax, {"raw": rep_raw, f"temperature-scaled (T={T:.2f})": rep_cal})
ax.set_title("Does 0.7 mean 70%?")
viz.save(fig, "g04_reliability")
plt.show()
cfg.append_result({"notebook": "04", "name": "calibration",
                   "temperature": float(T), "ece_raw": rep_raw["ece"],
                   "ece_calibrated": rep_cal["ece"]})

In [ ]:
# score EVERY analysable plot (including Hansen's ambiguous band) for chapter 05
models_all = []
for seed in cfg.SEEDS:
    m_ = M.SiameseChangeNet()
    m_.load_state_dict(torch.load(cfg.OUTPUT_DIR / f"cnn_seed{seed}.pt", map_location="cpu"))
    models_all.append(m_)

plot_rows = np.where(meta.kind.to_numpy() == "plot")[0]
loader_all = make_loader(plot_rows, False)
z_all = np.mean([M.evaluate(m_, loader_all)[1] for m_ in models_all], axis=0)
probs_all = 1 / (1 + np.exp(-z_all / T))

pd.DataFrame({"plot_id": meta.sample_id.to_numpy()[plot_rows],
              "model_prob": probs_all,
              "t1_item": "see chips.npz provenance",
              }).to_csv(cfg.OUTPUT_DIR / "cnn_predictions.csv", index=False)
print(f"calibrated probabilities saved for {len(plot_rows)} plots")

### What this chapter established

1. **Pixels beat plot means where it matters** — compare the PR curves at
   high recall: spatial texture lets the CNN separate partial clearings from
   intact forest that plot-mean series blur together (exact deltas in the
   ledger).
2. **The hard-negative ablation is the story**: removing TMF stable-forest
   negatives moved precision at 90% recall far more than any architecture
   decision made in this notebook. In this domain the training *data* is the
   model — which is also the published experience of the team this project
   is in dialogue with.
3. **Raw confidences were miscalibrated and one scalar fixed most of it** —
   necessary because a downstream verdict layer consumes these numbers as
   probabilities.

**Honest limitation, carried to chapter 06:** labels and hard negatives both
descend from Landsat-scale products; agreement with Hansen is partly
agreement *about* Hansen. The date-agreement check (ch. 03) and the
independent-map disagreement analysis (ch. 02) are the counterweights.

**Next:** everything becomes a verdict — fusion rules, tier maps, and the
evidence bundle an auditor would actually open.